In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import logomaker
import os
import scipy.stats
import multiprocessing
import matplotlib.gridspec as gridspec
import glob
import sys
import numpy as np

# Used colorpalette
black = '#000000'
lightblack = '#333333'
darkgray = '#666666'
mediumgray = '#999999'
lightgray = '#CCCCCC'

darkred = '#FF0000'
red = '#FF3333'
lightred = '#FF6666'
pink ='#FF9999'
salmon = '#FFCCCC'

In [2]:
print('########################################################################')
print('#### Make supp_database_1 of Gralak et al, ©Antoni Gralak_04.07.2025 ###')
print('########################################################################')
print('Setting env...')

num_cores = 23

def plot_messages(messages, ax):
    """
    Display messages in a textbox-like format on the given axis.
    """
    ax.axis('off')
    message_text = "\n".join(messages)
    ax.text(0, 1, message_text, fontsize=9, verticalalignment='top', family='monospace')

########################################################################
#### Make supp_database_1 of Gralak et al, ©Antoni Gralak_04.07.2025 ###
########################################################################
Setting env...


In [3]:
sys.path.append('/data/gralak/meSMiLEseq_github/meSMiLEseq')
import utils

In [4]:
# How many scatter point shall be filled
N_points_filled = 10
# how many to use for lin reg
N_lin_reg = 50 #100

In [5]:
TFs = os.listdir('/home/gralak/updepla/users/gralak/SmileSeq_paper/meSMiLEseq_motifs_and_scatterplots_for_publication')

In [6]:
TFs[0]

'ZBTB46_DBD'

In [29]:
for TF in TFs:
    data_path = f'/home/gralak/updepla/users/gralak/SmileSeq_paper/meSMiLEseq_motifs_and_scatterplots_for_publication/{TF}/'
    joint_matrices_path = os.path.join(data_path, 'joint/matrices/')
    
    sep_matrices_path = os.path.join(data_path, 'separated/matrices/')
    
    
    # Search and read all the data
    files_joint = os.listdir(joint_matrices_path)
    ratios = [f for f in files_joint if f.endswith('ratios.csv')]
    pvals = [f for f in files_joint if f.endswith('significant.csv')]
    dGGs = [f for f in files_joint if f.endswith('bindingmode_1.csv')]

    files_sep = os.listdir(sep_matrices_path)
    meths = [f for f in files_sep if f.endswith('_methylated_bindingmode_1.csv')]
    unmeths = [f for f in files_sep if f.endswith('_unmethylated_bindingmode_1.csv')]


    # Read the data

    ratio = pd.read_csv(os.path.join(joint_matrices_path, ratios[0]), index_col=0)
    if not pvals:
        pval = None
        significant_kmer_count = 0
        alert = 'No significant kmers!'
    else:
        try:
            pval = pd.read_csv(os.path.join(joint_matrices_path,pvals[0]))
            significant_kmer_count = len(pval)
            alert = None
        except FileNotFoundError:
            pval = None
            significant_kmer_count = 0
            alert = 'No significant kmers!'

    dGGj = pd.read_csv(os.path.join(joint_matrices_path, dGGs[0]), index_col=0)

    dGGm = pd.read_csv(os.path.join(sep_matrices_path, meths[0]), index_col=0)
    dGGum = pd.read_csv(os.path.join(sep_matrices_path, unmeths[0]), index_col=0)

    #Prepare alert messages
    messages = []
    if alert:
        messages.append(alert)
    elif significant_kmer_count < 20:
         messages.append("Poor enrichment (<20 significant kmers)")


    if pval is not None:
        # define how many kmers shall be filled out
        most_significant_kmers = ratio[ratio['significant'] ==  True].nsmallest(N_points_filled, ['pval_methl', 'pval_nonmethl'])
        kmer_filled = most_significant_kmers['index']
        ratio['fill_plot'] = ratio['index'].isin(kmer_filled)
        # Append kmer info to messages
        messages.append("Top significant kmers:")
        for _, row in most_significant_kmers.iterrows():
            messages.append(
                f"{row['index']}, x: {row['eluted_methl']:.3f}, y: {row['eluted_nonmethl']:.3f}"
        )
        

        kmer_lin_reg = ratio[ratio['significant'] ==  True].nsmallest(N_lin_reg, ['pval_methl', 'pval_nonmethl'])['index']
        ratio['use_for_linreg'] = ratio['index'].isin(kmer_lin_reg)

    else:
        ratio['significant'] = False
        ratio['use_for_linreg'] = False
        ratio['fill_plot'] = False

            

    lin_reg = {}
    coords = {}
    if pval is not None:
        for spec, dataframe in ratio.groupby(['use_for_linreg', 'CpG']):
            if spec[0]:
                if spec[1]:
                    key = 'with_CG'
                else:
                    key = 'no_CG'
                        
                dataframe_no_nan = dataframe.dropna(subset=['eluted_methl', 'eluted_nonmethl']) #drop nan otherwise no linreg possible
                if not dataframe_no_nan.empty:
                    x_l = dataframe_no_nan.eluted_methl
                    y_l = dataframe_no_nan.eluted_nonmethl

                    if len(x_l) < 4 or len(y_l) < 4 or x_l.nunique() == 1 or y_l.nunique() == 1:
                            continue
                    else:        
                        slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(x_l, y_l)
                        lin_reg[key] = [slope, intercept, r_value, p_value, std_err]
                        x_vals = np.linspace(x_l.min(), x_l.max(), 100)
                        y_vals = slope * x_vals + intercept
                        coords[key] = {'x': x_vals, 'y': y_vals}
    
    # Add regression slopes if available
    for key in lin_reg:
        slope = lin_reg[key][0]
        messages.append(f"Slope ({key}): {slope:.2f}")

    #######################################################################################################
    #Done with scatterplot, now motifs:
    dGGj_rev = utils.rev_complement(dGGj)
    dGGm_rev = utils.rev_complement(dGGm, extended_alphabet=False)
    dGGum_rev = utils.rev_complement(dGGum, extended_alphabet=False)

    # link slope with lettersize for joint motif

    stats_df = utils.calculate_stats2(dGGj)
    max_value_mg = pd.to_numeric(stats_df['mg'], errors='coerce')
    stats_df.to_csv(os.path.join(data_path, f'{TF}_logo_stats.csv'))
    
    stats_df_rev = utils.calculate_stats2(dGGj_rev)
    stats_df_rev.to_csv(os.path.join(data_path, f'{TF}_logo_rev_stats.csv'))
    

    # save mg or cg letter aaverage letter heights and the slopes. If methyl plus, there should be anticorrelation, if methyl minus, correlation
    letter_slope = {}
    try:
        letter_slope[TF] = {'mg': max_value_mg.max(), 'slope': lin_reg['with_CG'][0]}
    except KeyError:
        letter_slope[TF] = {'mg': max_value_mg.max(), 'slope': 'NA'}

    # PLOT THE SCATTERPLOT WITH FILLED KMERS AND SLOPES

    fig = plt.figure(figsize=(15, 12))
    gs = gridspec.GridSpec(4, 3, figure=fig) #width_ratios=[3.5, 1.5])


    # Define axes as per Excel layout
    ax_scatter = fig.add_subplot(gs[0:2, 0])  # Rows 0-1, Col 0 (scatterplot)

    # Top row joint logos and delta plots
    ax_dGGj = fig.add_subplot(gs[0, 1])       # Row 0, Col 1
    ax_dGGj_rev = fig.add_subplot(gs[0, 2])   # Row 0, Col 2
    ax_delta_dGGj = fig.add_subplot(gs[1, 1]) # Row 1, Col 1
    ax_delta_dGGj_rev = fig.add_subplot(gs[1, 2]) # Row 1, Col 2

    # Middle row: methylated / unmethylated logos
    ax_dGGm = fig.add_subplot(gs[2, 0])       # Row 2, Col 0
    ax_dGGum = fig.add_subplot(gs[2, 1])      # Row 2, Col 1
    ax_messages_1 = fig.add_subplot(gs[2, 2]) # Row 2, Col 2

    # Bottom row: reversed methylated / unmethylated logos
    ax_dGGm_rev = fig.add_subplot(gs[3, 0])   # Row 3, Col 0
    ax_dGGum_rev = fig.add_subplot(gs[3, 1])  # Row 3, Col 1
    ax_messages_2 = fig.add_subplot(gs[3, 2]) # Row 3, Col 2

    # Hide axes for messages (to be populated with text later)
    ax_messages_1.axis('off')
    ax_messages_2.axis('off')

    #Scatterplot
    #ax0 = fig.add_subplot(gs[0, 0])
    utils.kmer_scatterplots(ratio_df=ratio, coords=coords, ax=ax_scatter)
    ax_scatter.set_title(f'{TF} enrichment, normalized by input', fontsize=10)

    # Joint logo + delta
    utils.make_logo(dGGj, ax=ax_dGGj, title='Joint Logo')
    utils.plot_delta_mg_CG_context_aware(stats_df, ax=ax_delta_dGGj)

    utils.make_logo(dGGj_rev, ax=ax_dGGj_rev, title='Joint Logo (rev)')
    utils.plot_delta_mg_CG_context_aware(stats_df_rev, ax=ax_delta_dGGj_rev)

    # Methylated & Unmethylated Logos
    utils.make_logo(dGGm, ax=ax_dGGm, title='Methylated Logo')
    utils.make_logo(dGGm_rev, ax=ax_dGGm_rev, title='Methylated Logo (rev)')

    utils.make_logo(dGGum, ax=ax_dGGum, title='Unmethylated Logo')
    utils.make_logo(dGGum_rev, ax=ax_dGGum_rev, title='Unmethylated Logo (rev)')

    # Messages
    plot_messages(messages, fig.add_subplot(gs[2:, 2]))

    plt.tight_layout()
    #plt.show()
    plt.savefig(os.path.join(data_path, f'{TF}_final_plot.pdf'), dpi=300)
    plt.close()
    
    

In [28]:
if not pvals[0]:
    print('here')

IndexError: list index out of range

In [16]:
utils.rev_complement(dGGm, extended_alphabet=False)

,A,C,G,T
0,-0.464525,-0.305716,0.499896,-0.858934
1,-0.818897,-0.645073,1.050014,-0.715324
2,-0.718196,-0.623762,1.010278,-0.797599
3,-0.875604,-0.682866,0.983674,-0.554483
4,-0.320826,-0.109298,-0.432696,-0.266460
5,-0.237839,-0.704317,-0.273843,0.086720
6,-0.460384,0.108657,-1.049788,0.272236
7,0.242467,-0.975681,0.202418,-0.598484
8,0.294904,-0.347027,-0.607833,-0.469324
9,-0.393418,0.245741,-0.504289,-0.477314


In [35]:
most_significant_kmers = ratio[ratio['significant'] ==  True].nsmallest(N_points_filled, ['pval_methl', 'pval_nonmethl'])


In [45]:
dGGs

['ZBTB46_DBD_bindingmode_1.csv']

In [28]:
ratio[ratio['significant'] ==  True].nsmallest(10, ['pval_methl', 'pval_nonmethl'])['index']

80      AACCCC
1351    CCCCAA
1355    CCCCCA
2712    GGGGTT
336     ACCCCC
2367    GCCCCA
335     ACCCCA
1356    CCCCCC
3642    TGCCCC
3388    TCCCCC
Name: index, dtype: object

In [ ]:
def process_experiment(TF):
    data_path = f'/home/gralak/updepla/users/gralak/SmileSeq_paper/meSMiLEseq_motifs_and_scatterplots_for_publication/{TF}/'
    p_val_path = os.path.join(data_path, experiment_name, '02_fishers_exact_test/significant_kmers/')
    ratio_path = os.path.join(data_path, experiment_name, '03_kmer_ratios/')
    save_path = os.path.join(data_path, experiment_name, '03_kmer_ratios/with_slopes')

    os.makedirs(save_path, exist_ok=True)
        
    to_be_analyzed = [f for f in os.listdir(ratio_path) if f.endswith('csv')]

    for file in to_be_analyzed:
        if '9mer' not in file: 
            TF = file.split('_')[0] + '_' + file.split('_')[1]
            kmer = file.split('_')[2]
            ratios = pd.read_csv(os.path.join(ratio_path, file))
            try:
                p_val = pd.read_csv(os.path.join(p_val_path, f'{TF}_{kmer}_significant.csv'))
                significant_kmer_count = len(p_val)
                alert = None
            except FileNotFoundError:
                p_val = None
                significant_kmer_count = 0
                alert = 'No significant kmers!'
            

            #Prepare alert messages
            messages = []
            if alert:
                messages.append(alert)
            elif significant_kmer_count < 20:
                messages.append("Poor enrichment (<20 significant kmers)")


            if p_val is not None:
                # Calculate kmer slopes!
                pvals_m = p_val[p_val['mod'] == 'methl'][['kmer', 'p_adjust']].rename(columns={'p_adjust': 'pval_methl'})
                pvals_nm = p_val[p_val['mod'] == 'nonmethl'][['kmer', 'p_adjust']].rename(columns={'p_adjust': 'pval_nonmethl'})

                ratios = ratios.merge(pvals_m, left_on='index', right_on='kmer', how='left').drop(columns='kmer')
                ratios = ratios.merge(pvals_nm, left_on='index', right_on='kmer', how='left').drop(columns='kmer')

                
                significant_kmers = ratios[(ratios['pval_methl'] <= 0.5) | (ratios['pval_nonmethl'] <= 0.5)]['index']
                
                ratios['significant'] = ratios['index'].isin(significant_kmers)

                

                most_significant_kmers = ratios[ratios['significant'] ==  True].nsmallest(50, ['pval_methl', 'pval_nonmethl'])['index']
                ratios['fill_plot'] = ratios['index'].isin(most_significant_kmers)

                most_significant_kmers = ratios[ratios['significant'] ==  True].nsmallest(100, ['pval_methl', 'pval_nonmethl'])['index']
                ratios['use_for_linreg'] = ratios['index'].isin(most_significant_kmers)

            else:
                ratios['significant'] = False
                ratios['use_for_linreg'] = False
                ratios['fill_plot'] = False

            

            lin_reg = {}
            coords = {}
            if p_val is not None:
                for spec, dataframe in ratios.groupby(['use_for_linreg', 'CpG']):
                    if spec[0]:
                        if spec[1]:
                            key = 'with_CG'
                        else:
                            key = 'no_CG'
                        
                        dataframe_no_nan = dataframe.dropna(subset=['eluted_methl', 'eluted_nonmethl']) #drop nan otherwise no linreg possible
                        if not dataframe_no_nan.empty:
                            x_l = dataframe_no_nan.eluted_methl
                            y_l = dataframe_no_nan.eluted_nonmethl

                            if len(x_l) < 2 or len(y_l) < 2 or x_l.nunique() == 1 or y_l.nunique() == 1:
                                continue
                            else:        
                                slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(x_l, y_l)
                                lin_reg[key] = [slope, intercept, r_value, p_value, std_err]
                                x_vals = np.linspace(x_l.min(), x_l.max(), 100)
                                y_vals = slope * x_vals + intercept
                                coords[key] = {'x': x_vals, 'y': y_vals}
            

            # Add regression slopes if available
            for key in lin_reg:
                slope = lin_reg[key][0]
                messages.append(f"Slope ({key}): {slope:.2f}")

            ratios.to_csv(os.path.join(save_path, f'{TF}_{kmer}_ratios.csv'))
            
            # PLOT THE SCATTERPLOT WITH FILLED KMERS AND SLOPES


            fig = plt.figure(figsize=(9, 5.5))
            gs = gridspec.GridSpec(1, 2, width_ratios=[3.5, 1.5])
            ax0 = fig.add_subplot(gs[0])  # scatter plot
            ax1 = fig.add_subplot(gs[1])  # text box
            #fig, ax = plt.subplots(1, 1, figsize=(5.5, 5.5))

            x = ratios['eluted_methl']
            y = ratios['eluted_nonmethl']
            
            edgecolors = ratios['CpG'].map({True: darkred, False: lightblack})
            
            
            facecolors = [
                edgecolors.iloc[i] if sig else 'None'
                for i, sig in enumerate(ratios['fill_plot'])
            ]
            
            ax0.scatter(x=x, y=y, facecolors=facecolors, edgecolors=edgecolors, rasterized = True)
            if 'with_CG' in coords:
                ax0.plot(coords['with_CG']['x'], coords['with_CG']['y'], color=darkred)
            if 'no_CG' in coords:
                ax0.plot(coords['no_CG']['x'], coords['no_CG']['y'], color=lightblack)

            ax0.grid(visible=False)

            # Remove the top and right spines
            ax0.spines['top'].set_visible(False)
            ax0.spines['right'].set_visible(False)

            lower_limit = None
            upper_limit = max(x.max(), y.max()) * 1.05
            # Extent the axis by 5 % of max value
            ax0.set_xlim(lower_limit, upper_limit)
            ax0.set_ylim(lower_limit, upper_limit) 

            ax0.set_aspect('equal', adjustable='box')
                
                
            ax0.set_xlabel('methylated DNA', fontfamily='sans-serif', fontsize=10, fontstyle='italic')
            ax0.set_ylabel('unmethylated DNA', fontfamily='sans-serif', fontsize=10, fontstyle='italic')
                
            ax0.set_title(f"{TF} {kmer} enrichment, normalized by input", fontsize=10)

            # Display the messages in ax1
            ax1.axis('off')
            for i, msg in enumerate(messages):
                ax1.text(0, 1 - 0.1*i, msg, fontsize=9, fontfamily='monospace', va='top')
            
            plt.tight_layout()
            plt.savefig(os.path.join(save_path, f'{TF}_{kmer}_scatterplot.pdf'), dpi=400, bbox_inches='tight')
            plt.close()